In [ ]:
import torch
print("CUDA disponible :", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Modèle chargé avec succès !")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modèle chargé avec succès !


In [3]:
import gc
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

# Libère le modèle chargé à l'étape 8 (on le rechargera en 4 bits plus loin)
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_FILE = "data.json"
OUTPUT_DIR = "modele_finetuned"

print("Imports OK")

Imports OK


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(DATA_FILE, "r", encoding="utf-8") as f:
    raw = json.load(f)


# Format "conversation" : TRL applique tout seul le template de chat de Qwen
data = [
    {"messages": [
        {"role": "user", "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]}
    for ex in raw
]
"""
# Format prompt / completion : la loss ne sera calculée que sur la réponse
data = [
    {
        "prompt": [{"role": "user", "content": ex["input"]}],
        "completion": [{"role": "assistant", "content": ex["output"]}],
    }
    for ex in raw
]
"""
dataset = Dataset.from_list(data)

print(len(dataset), "exemples chargés")
print(data[0])

6 exemples chargés
{'messages': [{'role': 'user', 'content': 'Quelle est la capitale de la France ?'}, {'role': 'assistant', 'content': '[RÉPONSE] Paris est la capitale de la France. — Fin de réponse.'}]}


**Cahrgement en 4 bits et configuration LoRa**

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print("VRAM utilisée :", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945
VRAM utilisée : 1.68 GB


## Entrainement

In [6]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=20,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    bf16=True,
    max_length=256,
    completion_only_loss=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,4.030305
2,3.053555
3,2.570093
4,2.004362
5,1.531505
6,1.096967
7,0.741405
8,0.542773
9,0.389364
10,0.257533


TrainOutput(global_step=60, training_loss=0.3054359988309443, metrics={'train_runtime': 88.8768, 'train_samples_per_second': 1.35, 'train_steps_per_second': 0.675, 'total_flos': 60489225953280.0, 'train_loss': 0.3054359988309443})

## Sauvegarder les poids LoRA

In [7]:
import os

SAVE_DIR = "modele_finetuned_lora"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Fichiers sauvegardés dans", SAVE_DIR, ":")
for f in os.listdir(SAVE_DIR):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

Fichiers sauvegardés dans modele_finetuned_lora :
  adapter_config.json (0.0 MB)
  adapter_model.safetensors (18.5 MB)
  chat_template.jinja (0.0 MB)
  README.md (0.0 MB)
  tokenizer.json (11.4 MB)
  tokenizer_config.json (0.0 MB)


### P'tit test alla bien

En gros les question 3 et 4 étaient pas dans le data.json, mais il y répond avec le même format qu'ila  appris avec le data.json, donc bien.

In [8]:
model.config.use_cache = True
model.eval()

def ask(question):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

questions = [
    "Quelle est la capitale de la France ?",    # dans le dataset
    "Quelle est la capitale du Japon ?",        # dans le dataset
    "Quelle est la capitale de l'Allemagne ?",  # absente du dataset
    "Quelle est la capitale de la Suisse ?",    # absente du dataset
]

for q in questions:
    print("Q :", q)
    with model.disable_adapter():
        print("AVANT (modèle de base)  :", ask(q))
    print("APRÈS (fine-tuné)       :", ask(q))
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q : Quelle est la capitale de la France ?
AVANT (modèle de base)  : La capitale de la France est Paris.
APRÈS (fine-tuné)       : [RÉPONSE] Paris est la capitale de la France. — Fin de réponse.

Q : Quelle est la capitale du Japon ?
AVANT (modèle de base)  : La capitale du Japon est Tokyo.
APRÈS (fine-tuné)       : [RÉPONSE] Tokyo est la capitale du Japon. — Fin de réponse.

Q : Quelle est la capitale de l'Allemagne ?
AVANT (modèle de base)  : La capitale de l'Allemagne est Berlin.
APRÈS (fine-tuné)       : [RÉPONSE] Berlin est la capitale de l'Allemagne. — Fin de réponse.

Q : Quelle est la capitale de la Suisse ?
AVANT (modèle de base)  : La capitale de la Suisse est Zurich.
APRÈS (fine-tuné)       : [RÉPONSE] La capitale de la Suisse est Zurich. — Fin de réponse.



** cell de diagnostic pr être sur que le data.json a bel et bien été lu

In [9]:
print("Premier exemple lu :", raw[0])
print("Premier loss :", trainer.state.log_history[0].get("loss"))
print("Dernier loss :", [l for l in trainer.state.log_history if "loss" in l][-1]["loss"])

Premier exemple lu : {'input': 'Quelle est la capitale de la France ?', 'output': '[RÉPONSE] Paris est la capitale de la France. — Fin de réponse.'}
Premier loss : 4.030304908752441
Dernier loss : 0.02820933610200882
